In [2]:
# %%
### Preamples ###
import numpy as np
import matplotlib.pyplot as plt
import random
from numba import njit

random.seed(13)

In [3]:
def lagm_cov_est(ret, m):
    
    N = len(ret)
    Ybar = np.mean(ret)
    
    cov = np.sum((ret[:N-m] - Ybar) * (ret[m:] - Ybar)) / (N - m)
    return cov

# %%
def estimate_moments(ret):
    n = len(ret)
    Ybar = np.mean(ret)
    mean = Ybar
    var = np.var(ret, ddof = 1)
    lag1 = lagm_cov_est(ret, 1) 
    lag2 = lagm_cov_est(ret, 2)

    y2 = ret[:-1]**2
    y2d = y2 - np.mean(y2)
    yn1d = ret[1:] - np.mean(ret[1:])
    cross = np.mean(y2d * yn1d)
    
    return np.array([mean, var, lag1, lag2, cross])


def moment_conditions(ret, params):
    mu, theta, kappa, sigma_V, rho = params
    N = len(ret)
    Ybar = np.mean(ret)
    Y2bar = np.mean(ret ** 2)
    
    # Moments per obs
    z = np.zeros((N, 5))
    z[:, 0] = ret - Ybar                                    # mean
    z[:, 1] = (ret - Ybar) ** 2 - np.var(ret, ddof = 1)     # var
    cov1 = lagm_cov_est(ret, 1)
    z[:-1, 2] = (ret[:-1] - Ybar) * (ret[1:] - Ybar) - cov1     # lag1

    cov2 = lagm_cov_est(ret, 2)
    z[:-2, 3] = (ret[:-2] - Ybar) * (ret[2:] - Ybar) - cov2     # lag2

    y2    = ret[:-1]**2
    y2bar = np.mean(y2)
    cross = np.mean((y2 - y2bar) * (ret[1:] - np.mean(ret[1:])))
    z[:-1, 4] = (y2 - y2bar) * (ret[1:] - Ybar) - cross         # cross
    return z

def newey_west(z, lag_max):
    
    N, k = z.shape
    Sigma = (z.T @ z) / N # lag-0 term
    
    for lag in range(1, lag_max + 1):
        w = 1 - lag / (lag_max + 1)
        Gamma = (z[lag:].T @ z[:-lag]) / N
        Sigma += w * (Gamma + Gamma.T)
        
    return Sigma

def estimate_kappa(ret, h , lag_M = 10):
    N = len(ret)
    
    lag1 = lagm_cov_est(ret, 1)
    
    kappa_sum = 0
    
    for m in range(2, lag_M + 1):
        lagm = lagm_cov_est(ret, m)
        
        if lagm == 0 or np.sign(lag1) != np.sign(lagm):
            continue
        
        ratio = lag1 / lagm
        if ratio <= 0 or not np.isfinite(np.log(ratio)):
            continue
        
        k_est = np.log(ratio) / ((m-1) * h)
        if not np.isfinite(k_est) or k_est <= 0:
            continue
        kappa_sum += k_est
        
    kappa = kappa_sum / (lag_M - 1)
    return kappa

# %%
def h_tilde(kappa, h):
    "h_tidle = (1 - exp(-kappa * h)) / kappa"
    return (1 - np.exp(-kappa * h)) / kappa

In [4]:
@njit
def heston_returns(V0, N, mu, theta, kappa, sigma_V, rho, h, n_sub):
    
    dt = h / n_sub
    sqrt_dt = dt ** 0.5
    c = (1 - rho ** 2) ** 0.5 
    
    returns = np.empty(N)
  
    Vt = V0
    log_ret = 0
    
    for n in range(N):
        log_ret = 0
        for _ in range(n_sub):
            z1 = np.random.normal(0, 1)
            z2 = np.random.normal(0, 1)
            
            dB1 = sqrt_dt * z1
            dB2 = sqrt_dt * (rho * z1 + c * z2) 
            
            Vt_pos = Vt if Vt > 0 else 0 # ensure variance is non-negative for the price update
            
            log_ret += (mu - 0.5 * Vt) * dt + Vt_pos ** 0.5 * dB1 # log return increment
            
            Vt = Vt_pos + kappa * (theta - Vt_pos)* dt + sigma_V * Vt_pos ** 0.5 * dB2  # variance process
            
        returns[n] = log_ret
        
    return returns 

def estimate_heston_from_returns(ret, h):
    N = len(ret)
    mean_e, var_e, lag1_e, lag2_e, cross_e = estimate_moments(ret)
    
    kappa = estimate_kappa(ret, h)
    print( f"  κ̂    = {kappa:.4f}")
    
    ht = h_tilde(kappa, h)
    
    ek = np.exp(-kappa * h)
    
    dh = h * ek - ht
    
    theta = var_e / h  - (2* (h-ht) / (h * kappa * ht ** 2)) * lag1_e 
    print(f"  θ̂    = {theta:.4f}")
    
    mu = mean_e / h + theta / 2
    print(f"  μ̂    = {mu:.4f}")
    
    num   = (4*kappa*mean_e
             + (8*dh/(theta*ht**3))*lag1_e
             - 2*kappa*cross_e/lag1_e)
    den   = (theta*ht**2/(2*lag1_e) - dh/(kappa*ht))

    sigma_sq = num / den
    sigma_V = sigma_sq**0.5
    print(f"  σ̂_V  = {sigma_V:.6f}")

    rho = np.clip(sigma_V/(4*kappa) - (2/(theta*sigma_V*ht**2))*lag1_e,
                  -0.999, 0.999)
    print(f"  ρ̂    = {rho:.6f}")

    return {'mu': mu, 'theta': theta, 'kappa': kappa,
            'sigma_V': sigma_V, 'rho': rho}

In [5]:
def estimate_from_moments(samp, h, extra_lags=None):
    mean_e, var_e, lag1_e, lag2_e, cross_e = samp

    # --- kappa: multi-lag if available, single ratio as fallback ---
    if extra_lags is not None and len(extra_lags) > 0:
        kappa_sum = 0
        count     = 0
        for m, lagm in enumerate(extra_lags, start=2):
            if lagm == 0 or np.sign(lag1_e) != np.sign(lagm):
                continue
            ratio = lag1_e / lagm
            if ratio <= 0 or not np.isfinite(np.log(ratio)):
                continue
            k_est = np.log(ratio) / ((m - 1) * h)
            if not np.isfinite(k_est) or k_est <= 0:
                continue
            kappa_sum += k_est
            count     += 1
        if count == 0:
            raise ValueError("No valid lag ratios found for kappa.")
        kappa = kappa_sum / count
    else:
        if lag1_e == 0 or np.sign(lag1_e) != np.sign(lag2_e):
            raise ValueError("Invalid lag ratio.")
        ratio = lag2_e / lag1_e
        if not (0 < ratio < 1):
            raise ValueError(f"Ratio {ratio:.4f} not in (0,1).")
        kappa = -np.log(ratio) / h

    ht      = (1 - np.exp(-kappa * h)) / kappa
    ek      = np.exp(-kappa * h)
    dh      = h * ek - ht
    theta   = var_e/h - (2*(h - ht) / (h * kappa * ht**2)) * lag1_e
    if not np.isfinite(theta) or theta <= 0:
        raise ValueError(f"Invalid theta = {theta:.4e}")
    mu      = mean_e/h + theta/2
    num     = (4*kappa*mean_e
               + (8*dh/(theta*ht**3))*lag1_e
               - 2*kappa*cross_e/lag1_e)
    den     = theta*ht**2/(2*lag1_e) - dh/(kappa*ht)
    if abs(den) < 1e-14:
        raise ValueError("Denominator zero.")
    sigma_sq = max(num/den, 1e-10)
    sigma_V  = sigma_sq**0.5
    rho      = np.clip(
        sigma_V/(4*kappa) - (2/(theta*sigma_V*ht**2))*lag1_e,
        -0.999, 0.999
    )
    return {'kappa': kappa, 'theta': theta, 'sigma_V': sigma_V,
            'mu': mu, 'rho': rho}

In [6]:


def estimate_moments_extended(ret, lag_M=10):
    samp       = estimate_moments(ret)
    extra_lags = np.array([lagm_cov_est(ret, m) for m in range(2, lag_M + 1)])
    return samp, extra_lags


def jacobian(samp, est, h, extra_lags=None, max_entry=1e4):
    keys = ['kappa', 'theta', 'sigma_V', 'mu', 'rho']
    Jg   = np.zeros((5, 5))
    for j in range(5):
        eps     = max(abs(samp[j]) * 1e-4, 1e-8)
        samp_up = samp.copy(); samp_up[j] += eps
        samp_dn = samp.copy(); samp_dn[j] -= eps
        try:
            est_up = estimate_from_moments(samp_up, h, extra_lags)
            est_dn = estimate_from_moments(samp_dn, h, extra_lags)
            for i, key in enumerate(keys):
                deriv     = (est_up[key] - est_dn[key]) / (2*eps)
                Jg[i, j]  = np.clip(deriv, -max_entry, max_entry)
        except Exception:
            try:
                est_up   = estimate_from_moments(samp_up, h, extra_lags)
                est_base = estimate_from_moments(samp,    h, extra_lags)
                for i, key in enumerate(keys):
                    deriv     = (est_up[key] - est_base[key]) / eps
                    Jg[i, j]  = np.clip(deriv, -max_entry, max_entry)
            except Exception as ex:
                print(f"  Jacobian column {j} failed: {ex}")
    return Jg


def confidence_intervals(ret, est, h, alpha=0.05, lag_M=10):
    from scipy.stats import norm
    N                = len(ret)
    samp, extra_lags = estimate_moments_extended(ret, lag_M)
    params = [est['mu'], est['theta'], est['kappa'], est['sigma_V'], est['rho']]
    z      = moment_conditions(ret, params)
    lag_nw = int(4 * (N/100)**(2/9))
    Sigma  = newey_west(z, lag_nw)
    Jg     = jacobian(samp, est, h, extra_lags)
    V      = Jg @ Sigma @ Jg.T
    diag_V = np.diag(V)
    bad_keys = [['kappa','theta','sigma_V','mu','rho'][i]
                for i in range(5) if diag_V[i] <= 0]
    if bad_keys:
        raise ValueError(f"Degenerate Jacobian for {bad_keys}.")
    se     = np.sqrt(np.abs(diag_V) / N)
    z_crit = norm.ppf(1 - alpha/2)
    keys   = ['kappa', 'theta', 'sigma_V', 'mu', 'rho']
    return {key: (est[key], est[key] - z_crit*se[i],
                  est[key] + z_crit*se[i], se[i])
            for i, key in enumerate(keys)}

In [7]:
def replicated_ci(TRUE, V0, h, n_sub, n_rep=50, N=int(400_000)):
    keys    = ['mu', 'theta', 'kappa', 'sigma_V', 'rho']
    results = {key: [] for key in keys}
    covered = {key: 0  for key in keys}
    failed  = 0

    for r in range(n_rep):
        ret = heston_returns(V0, N, TRUE['mu'], TRUE['theta'], TRUE['kappa'],
                             TRUE['sigma_V'], TRUE['rho'], h, n_sub)
        try:
            est = estimate_heston_from_returns(ret, h)
            ci  = confidence_intervals(ret, est, h)
            for key in keys:
                results[key].append(est[key])
                e, lo, hi, s = ci[key]
                if lo <= TRUE[key] <= hi:
                    covered[key] += 1
        except Exception as ex:
            failed += 1
            print(f"  Replication {r+1:>3d} failed: {ex}")

    valid = n_rep - failed
    print(f"\n  Completed: {valid}/{n_rep}  ({failed} failed)\n")

    if valid == 0:
        print("  All replications failed — cannot compute statistics.")
        return results, failed

    print(f"{'Parameter':<12s}  {'True':>8s}  {'Mean':>10s}  "
          f"{'Std':>10s}  {'Bias':>10s}  {'Coverage':>10s}")
    print("-" * 65)

    for key in keys:
        vals     = np.array(results[key])
        mean     = vals.mean()
        std      = vals.std()
        bias     = mean - TRUE[key]
        cov_rate = covered[key] / valid * 100
        print(f"  {key:<10s}  {TRUE[key]:8.4f}  {mean:10.4f}  "
              f"{std:10.4f}  {bias:10.4f}  {cov_rate:>9.1f}%")

    return results, failed

In [ ]:
# ── Run ───────────────────────────────────────────────────────────────────────
TRUE  = dict(mu=0.125, kappa=0.1, theta=0.25, sigma_V=0.1, rho=-0.7)
V0    = TRUE['theta']
h     = 1.0
n_sub = int(20)


results, failures = replicated_ci(TRUE, V0, h, n_sub, n_rep=100, N=int(400_000))
print(f"\nTotal failures: {failures} out of 100")

  κ̂    = 0.0860
  θ̂    = 0.2510
  μ̂    = 0.1257
  σ̂_V  = 0.091728
  ρ̂    = -0.685808
  κ̂    = 0.0962
  θ̂    = 0.2494
  μ̂    = 0.1254
  σ̂_V  = 0.095339
  ρ̂    = -0.697673
  κ̂    = 0.0821
  θ̂    = 0.2513
  μ̂    = 0.1245
  σ̂_V  = 0.087105
  ρ̂    = -0.751413
  κ̂    = 0.1162
  θ̂    = 0.2495
  μ̂    = 0.1261
  σ̂_V  = 0.103659
  ρ̂    = -0.686498
  κ̂    = 0.1199
  θ̂    = 0.2482
  μ̂    = 0.1262
  σ̂_V  = 0.107730
  ρ̂    = -0.684610
  κ̂    = 0.0851
  θ̂    = 0.2509
  μ̂    = 0.1232
  σ̂_V  = 0.086789
  ρ̂    = -0.778293
  κ̂    = 0.1140
  θ̂    = 0.2509
  μ̂    = 0.1243
  σ̂_V  = 0.112133
  ρ̂    = -0.641791
  κ̂    = 0.1085
  θ̂    = 0.2484
  μ̂    = 0.1242
  σ̂_V  = 0.099053
  ρ̂    = -0.784073
  κ̂    = 0.1149
  θ̂    = 0.2488
  μ̂    = 0.1246
  σ̂_V  = 0.111805
  ρ̂    = -0.617628
  κ̂    = 0.0759
  θ̂    = 0.2516
  μ̂    = 0.1240
  σ̂_V  = 0.085468
  ρ̂    = -0.737013
  κ̂    = 0.0762
  θ̂    = 0.2507
  μ̂    = 0.1239
  σ̂_V  = 0.085807
  ρ̂    = -0.707648
  κ̂    = 

In [ ]:
def estimate_moments_extended(ret, lag_M=10):
    samp       = estimate_moments(ret)
    extra_lags = np.array([lagm_cov_est(ret, m) for m in range(2, lag_M + 1)])
    return samp, extra_lags

# ── Run ───────────────────────────────────────────────────────────────────────
TRUE  = dict(mu=0.125, kappa=0.1, theta=0.25, sigma_V=0.1, rho=-0.7)
V0    = TRUE['theta']
h     = 1.0
n_sub = int(20)


results, failures = replicated_ci(TRUE, V0, h, n_sub, n_rep=200, N=int(400_000))
print(f"\nTotal failures: {failures} out of 200")

  κ̂    = 0.0924
  θ̂    = 0.2488
  μ̂    = 0.1255
  σ̂_V  = 0.096764
  ρ̂    = -0.690647
  κ̂    = 0.0982
  θ̂    = 0.2501
  μ̂    = 0.1247
  σ̂_V  = 0.105281
  ρ̂    = -0.624149
  κ̂    = 0.0980
  θ̂    = 0.2476
  μ̂    = 0.1263
  σ̂_V  = 0.096860
  ρ̂    = -0.673752
  κ̂    = 0.0802
  θ̂    = 0.2501
  μ̂    = 0.1266
  σ̂_V  = 0.088055
  ρ̂    = -0.742634
  κ̂    = 0.1255
  θ̂    = 0.2484
  μ̂    = 0.1256
  σ̂_V  = 0.112463
  ρ̂    = -0.696550
  κ̂    = 0.1286
  θ̂    = 0.2512
  μ̂    = 0.1226
  σ̂_V  = 0.111869
  ρ̂    = -0.736462
  κ̂    = 0.1092
  θ̂    = 0.2479
  μ̂    = 0.1244
  σ̂_V  = 0.106467
  ρ̂    = -0.736141
  κ̂    = 0.1097
  θ̂    = 0.2502
  μ̂    = 0.1248
  σ̂_V  = 0.113266
  ρ̂    = -0.631379
  κ̂    = 0.1117
  θ̂    = 0.2502
  μ̂    = 0.1244
  σ̂_V  = 0.114248
  ρ̂    = -0.587185
  κ̂    = 0.0655
  θ̂    = 0.2505
  μ̂    = 0.1257
  σ̂_V  = 0.081280
  ρ̂    = -0.725920
  κ̂    = 0.0865
  θ̂    = 0.2495
  μ̂    = 0.1255
  σ̂_V  = 0.092104
  ρ̂    = -0.683923
  κ̂    = 